In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model hazir!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\MSI-NB\Projects\CiteOrAbstain\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI-NB\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model hazir!


In [2]:
vector = model.encode("Knee replacement recovery")

print("Kac sayi var:", len(vector))
print("Ilk 5 sayi:", vector[:5])

Kac sayi var: 384
Ilk 5 sayi: [-0.11041609  0.04919952  0.0236715  -0.03717305 -0.01708467]


In [3]:
question = "When can I walk after knee surgery?"

passages = [
    "Most patients begin moving around with a walker the day after the operation.",
    "Physical therapy helps strengthen the muscles around your new knee.",
    "Knee replacement is a common procedure for severe arthritis.",
    "The weather in London is rainy today.",
]

question_vector = model.encode(question)
passage_vectors = model.encode(passages)

scores = model.similarity(question_vector, passage_vectors)[0]

for passage, score in zip(passages, scores):
    print(f"{float(score):.2f}  {passage}")

0.59  Most patients begin moving around with a walker the day after the operation.
0.53  Physical therapy helps strengthen the muscles around your new knee.
0.51  Knee replacement is a common procedure for severe arthritis.
-0.01  The weather in London is rainy today.


In [4]:
risky_question = "How much ibuprofen should I take after knee surgery?"

risky_vector = model.encode(risky_question)
risky_scores = model.similarity(risky_vector, passage_vectors)[0]

for passage, score in zip(passages, risky_scores):
    print(f"{float(score):.2f}  {passage}")

0.30  Most patients begin moving around with a walker the day after the operation.
0.36  Physical therapy helps strengthen the muscles around your new knee.
0.46  Knee replacement is a common procedure for severe arthritis.
-0.01  The weather in London is rainy today.


# Deneyden Çıkan Sonuçlar

## 1. Embedding nedir?
Embedding, bir cümleyi anlamını taşıyan sayılara çevirmektir. Kullandığımız model, her cümleyi uzunluğu ne olursa olsun **384 sayıya** çeviriyor. Tek bir sayının anlamı yok; anlam, bütün sayıların birlikte oluşturduğu konumda saklı.

## 2. Embedding kelimelere değil anlama bakar
"Diz ameliyatından sonra ne zaman yürüyebilirim?" sorusuna en yüksek skoru (**0.59**), soruyla neredeyse hiç ortak kelimesi olmayan "hastalar ameliyatın ertesi günü yürüteçle hareket etmeye başlar" pasajı aldı. Normal kelime araması bu pasajı bulamazdı.

## 3. Alakasız sorularda karar vermek kolay
Konuyla hiç ilgisi olmayan pasaj ("Londra'da hava yağmurlu") **-0.01** aldı. Konu dışı bir soru geldiğinde skorlar çok düşük çıkar ve sistem kolayca cevap vermekten kaçınır.

## 4. Asıl zorluk: Eşik değerini seçmek
Soruyu gerçekten cevaplayan pasaj **0.59** aldı. Konuyla ilgili ama soruyu cevaplamayan pasajlar ise **0.53** ve **0.51** aldı. Skorlar birbirine çok yakın. Yani embedding, "aynı konudan bahsediyor" ile "soruyu cevaplıyor" arasındaki farkı tam olarak ayırt edemiyor. Eşik değerini doğru ayarlamak projenin en kritik işi.

## 5. Eşik değeri tek başına tehlikeli soruları durduramaz
"Diz ameliyatından sonra ne kadar ibuprofen almalıyım?" gibi bir ilaç dozu sorusu bile **0.46** skor aldı. Gerçek belgelerde ilaçla ilgili pasajlar olacağı için bu skor daha da yüksek çıkabilir. Bu yüzden ilaç ve acil durum soruları için **skordan bağımsız** ve **aramadan önce** çalışan ayrı bir güvenlik kuralı gerekiyor.

## 6. Mimariye etkisi
- Önce soru kontrol edilir: İlaç ya da acil durumla mı ilgili?
- **Evet ise:** Skora bakılmadan sabit ret mesajı verilir.
- **Hayır ise:** Embedding, arama, benzerlik skoru ve eşik kontrolü yapılır. Skor eşiğin üstündeyse kaynak gösterilerek cevap verilir, altındaysa sistem cevap vermekten kaçınır.